In [ ]:
# packages
from pathlib import Path
import ssl
from urllib import request
import ppxf.sps_util as lib
import os
import numpy as np
from astropy.io import fits
import matplotlib.pyplot as plt

from importlib import resources

from ppxf.ppxf import ppxf
from ppxf.ppxf_util import log_rebin, vac_to_air, air_to_vac
import ppxf.ppxf_util as util
from astropy.wcs import WCS

import specutils

base = Path.cwd()

# speed of light
c = 299792.458 # km/s (everything ppxf outputs is in km/s)

In [ ]:
# Load the FITS file
obj = 'DESJ2112' # name of obj
AGEL_name = 'AGEL2112'
filter = 'r' # red-side filter of KCWI
type = 'etg'
option = 'qfits'
lines = True

"""keep for now"""
ra_of_interest = 318.183 # RA of spaxel of interest (degrees), use for plotting if qfits
dec_of_interest = 0.156 # Dec of spaxel of interest (degrees)


# initial guess for redshift (based on location of Na D doublet)
z_guess = 0.4436
apply_redshift = True

# Construct the file path

file_path = os.path.join(base, f'../data/{obj}_{filter}.fits')
if option == 'qfits':
    qfits_file_path = os.path.join(base, f'../data/{obj}_perturber_{filter}.fits')

with fits.open(file_path) as hdul:
    # Print the header information
    hdr = hdul[0].header
    print(hdr)
    # Extract the data from the FITS file
    cube = hdul[0].data
    wcs = WCS(hdr)

# extract everything from the cube
ref_lambda = hdr['CRVAL3'] # wavelength zeropoint (beginning wavelength)
if filter == 'r':
    delt = hdr['CD3_3'] # wavelength step size
else:
    delt = hdr['CDELT3'] # wavelength step size (1.0 angstroms per pixel)
npix = hdr['NAXIS3'] # number of wavelength pixels
pix = np.arange(npix) # pixel indices
crpix = hdr.get('CRPIX3', 1.0) # reference pixel (default to 1.0 if not present)
wavelengths = ref_lambda + (pix - crpix) * delt # wavelength array

#spaxel_of_interest = (51, 62) # central spaxel coordinate (y,x)


In [ ]:
# Construct position array for spaxels, or use qfitsview fits file

if option == 'qfits':
    with fits.open(qfits_file_path) as hdul_qfits:
        # Print the header information
        hdr_qfits = hdul_qfits[0].header
        print(hdr_qfits)
        # Extract the data from the FITS file
        cube_qfits = hdul_qfits[0].data
        wcs = WCS(hdr)

    """fix later
    ra_of_interest = wcs.
    dec_of_interest
    """
    deflector_spectrum_raw = cube_qfits
    

elif option == 'custom':
    
    # Convert WCS sky coords -> cube spaxel indices
    x_pix, y_pix = wcs.celestial.world_to_pixel_values(ra_of_interest, 
                                                    dec_of_interest)
    x_idx = int(np.round(x_pix))
    y_idx = int(np.round(y_pix))
    
    # Create aperture of radius 3 spaxels
    aperture_radius = 3
    yy, xx = np.ogrid[-aperture_radius:aperture_radius+1, -aperture_radius:aperture_radius+1]
    mask = (xx**2 + yy**2) <= aperture_radius**2
    y_indices = y_idx + np.where(mask)[0] - aperture_radius
    x_indices = x_idx + np.where(mask)[1] - aperture_radius

    # obtain deflector spectrum at the spaxel of interest
    deflector_spectrum_raw = np.median(cube[:, y_indices, x_indices], axis=1)  # median across spaxels in aperture
    #deflector_spectrum_raw = np.average(cube[:, y_idx, x_idx], axis = (1,2))


# obtain noise estimate
noise_raw = np.std(cube[:, 30:35, 65:71], axis=(1, 2)) - np.median(cube[:, 30:35, 65:71], axis=(1, 2)) # change the coords when needed

In [ ]:
#  =======
# Construct array and smoothen spectrum
# ============

# construct the wavelength array in vacuum wavelengths
wavelength_vac = ref_lambda + delt * np.arange(npix) # in angstrom, ref_lambda is start, delt is step size, npix is number of pixels
wavelength_air = vac_to_air(wavelength_vac) # convert to air wavelengths, send to rest frame
print(wavelength_air[0] / wavelength_vac[0])

# Gaussian smoothing (in pixels)
sigma = 3
half = int(3 * sigma)
x = np.arange(-half, half + 1)
kernel = np.exp(-0.5 * (x / sigma) ** 2)
kernel /= kernel.sum()

# apply gaussian filter to deflector and noise spectra
deflector_spectrum = np.convolve(deflector_spectrum_raw, kernel, mode='same') / np.median(deflector_spectrum_raw) # normalize by median to get relative flux
noise_raw = np.convolve(noise_raw, kernel, mode='same') / np.median(deflector_spectrum_raw) # normalize noise by same factor as spectrum

spectrum_range = deflector_spectrum[0:2700]
max_line = np.argmax(spectrum_range) + ref_lambda / delt
print(max_line)

In [ ]:
# ====================
# Visualize the spectrum
# ===================

# common rest-frame lines (angstrom)
rest_emission_lines_ltg = {
    'Lyα': 1215.67,
    'N V': 1240.81,
    'C IV': 1549.48,
    'He II': 1640.42,
    'C III': 1907,
    'O II': 3727,
    #'[Ne III]': 3868.76,
    #'He I': 3889,
    'Hdelta': 4101.74,
    'Hgamma': 4340.47,
    #'He II 4685': 4685.68,
    'Hβ': 4861.33,
    'O III Doublet': 5007,
    'He II 5411': 5411.52,
    '[O I] 5577': 5577.339,
    '[N II] 5755': 5754.59,
    'Hα': 6562.8}

# simple color map for plotting
# emission = green, absorption = orange
rest_absorption_lines_ltg = {
    'Si II': 1260.42,
    'C II': 1334.53,
    '': 2796,
    'Fe II': 1608,
    'Mg II': 2803}

rest_emission_lines_etg = {'Hβ': 4861.33,
    'O III': 5007,
    'Hα': 6562.8,
    'N II i': 6548.05,
    'N II ii': 6583.45}

rest_absorption_lines_etg = {"": 3933.7, "Ca K + H" : 3968.5, "Ca G" : 
                             4307.74, "Mg I" : 5175.0, "Na": 5895.92}
    

line_colors_ltg = {
    name: ('tab:blue' if name in rest_absorption_lines_ltg else 'tab:orange')
    for name in rest_emission_lines_ltg
}

line_colors_etg = {
    name: ('tab:blue' if name in rest_absorption_lines_etg else 'tab:orange')
    for name in rest_emission_lines_etg
}

# compute plotted wavelength (rest or redshifted)
def plotted_lambda(rest_wl):
    z = z_guess if apply_redshift else 0.0
    return rest_wl * (1.0 + z)

# plot observed-frame spectrum with line locations
plt.figure(figsize=(20, 12))
fig, ax = plt.subplots(figsize=(20, 12))
plt.xlabel('Wavelength ($\\AA$, air)', fontsize=40)
plt.ylabel('Flux', fontsize=40)
np.median(deflector_spectrum_raw)
if filter == 'r':
    plt.xlim(5650, 8750)
elif filter == 'b':
    plt.xlim(3700, 5200)
    
plt.ylim(-1, 4)

# show all lines if lines configured
if lines == True:
    if type == 'ltg':
        line_colors = line_colors_ltg
        show_lines = set(rest_emission_lines_ltg.keys()) | set(rest_absorption_lines_ltg.keys())
        for name in show_lines:
            wl = plotted_lambda(rest_emission_lines_ltg.get(name, rest_absorption_lines_ltg.get(name)))
            if wl <= plt.xlim()[1] and wl >= plt.xlim()[0]: # only show lines within x-axis limits
                plt.axvline(wl, color=line_colors.get(name), linestyle='--', lw=2)
                plt.text(wl, plt.ylim()[1] * 0.8, name, rotation=45, fontsize=20, color = line_colors.get(name), verticalalignment='top')
    elif type == 'etg':
        line_colors = line_colors_etg
        show_lines = set(rest_emission_lines_etg.keys()) | set(rest_absorption_lines_etg.keys())
        print(show_lines)
        for name in show_lines:
            wl = plotted_lambda(rest_emission_lines_etg.get(name, rest_absorption_lines_etg.get(name)))
            if wl <= plt.xlim()[1] and wl >= plt.xlim()[0]: # only show lines within x-axis limits
                plt.axvline(wl, color=line_colors.get(name), linestyle='--', lw=2)
                plt.text(wl, plt.ylim()[1] * 0.8, name, rotation=45, fontsize=20, color = line_colors.get(name), verticalalignment='top')


plt.plot(wavelength_air, deflector_spectrum, color='green', lw=2, label='KCWI red-side spectrum', drawstyle='steps-mid')
plt.fill_between(wavelength_air, deflector_spectrum - noise_raw / 2, deflector_spectrum + noise_raw / 2, color='green', alpha=0.3, label='Noise estimate')
print(np.argmin(deflector_spectrum) + ref_lambda / delt)
plt.xlabel(r'$\lambda_{\rm{obs}}$ ($\AA$, air)', fontsize=40)
plt.ylabel('Flux', fontsize=40)
# end selection branch for filter-specific x-limits
plt.title(f"KCWI {filter}-side for {obj} ({ra_of_interest:.3f}, {dec_of_interest:.3f})", fontsize = 45)
plt.text(0.9*plt.xlim()[1], plt.ylim()[1] * - 0.3, r"$z_{\rm{prelim}} =$" f"{z_guess:.3f}", fontsize=35)
ax.tick_params(labelsize=20)
plt.legend(fontsize = 25, loc = 'upper left')

print(np.median(deflector_spectrum))